# Transfer Volume (Dashboard Migration)

Copies the source export volume contents to the target import volume (same metastore). Run on the **target** workspace after the source has written inventory, exported, and transformed artifacts.

## Configuration

Read widget parameters: source/target catalogs, source/target schemas, and volume names. Derive the export and import volume paths.

In [ ]:
dbutils.widgets.text("source_catalog", "", "Source catalog (holds export volume)")
dbutils.widgets.text("target_catalog", "", "Target catalog (holds import volume)")
dbutils.widgets.text("source_schema", "default", "Source schema")
dbutils.widgets.text("target_schema", "", "Target schema (defaults to source_schema)")
dbutils.widgets.text("export_volume", "dashboard_migration", "Export volume name")
dbutils.widgets.text("import_volume", "dashboard_migration", "Import volume name")

SOURCE_CATALOG = dbutils.widgets.get("source_catalog").strip()
TARGET_CATALOG = dbutils.widgets.get("target_catalog").strip()
SOURCE_SCHEMA = dbutils.widgets.get("source_schema").strip()
TARGET_SCHEMA = dbutils.widgets.get("target_schema").strip() or SOURCE_SCHEMA
EXPORT_VOLUME = dbutils.widgets.get("export_volume").strip()
IMPORT_VOLUME = dbutils.widgets.get("import_volume").strip()

if not SOURCE_CATALOG or not TARGET_CATALOG:
    raise ValueError("Set source_catalog and target_catalog")

SRC_PATH = f"/Volumes/{SOURCE_CATALOG}/{SOURCE_SCHEMA}/{EXPORT_VOLUME}"
TGT_PATH = f"/Volumes/{TARGET_CATALOG}/{TARGET_SCHEMA}/{IMPORT_VOLUME}"
print(f"Source: {SRC_PATH}")
print(f"Target: {TGT_PATH}")

## Verify export data exists

Checks that the export volume exists and contains at least one expected subdirectory (transformed or exported) before proceeding.

In [ ]:
expected_subdirs = ["transformed", "exported", "dashboard_inventory_approved", "dashboard_inventory"]

try:
    src_contents = [f.name.rstrip("/") for f in dbutils.fs.ls(SRC_PATH)]
    has_data = any(d in src_contents for d in expected_subdirs)
except Exception:
    has_data = False

if not has_data:
    print("No export data at source; skipping transfer.")
    dbutils.notebook.exit("SKIP_NO_SOURCE")

print(f"Found source data: {src_contents}")

## Create import volume and copy

Creates the import volume if needed, clears the target path, then copies all export files and subdirs to the target volume.

In [ ]:
spark.sql(f"CREATE VOLUME IF NOT EXISTS {TARGET_CATALOG}.{TARGET_SCHEMA}.{IMPORT_VOLUME}")

dbutils.fs.rm(TGT_PATH, recurse=True)

dbutils.fs.cp(SRC_PATH, TGT_PATH, recurse=True)

tgt_contents = [f.name for f in dbutils.fs.ls(TGT_PATH)]
print(f"TRANSFERRED to {TGT_PATH}")
print(f"Contents: {tgt_contents}")